# Cats vs Dogs — Entraînement sur Google Colab

Ce notebook clone le dépôt GitHub, installe les dépendances, prépare les données puis lance `train_cats_dogs.py`.


## Étape 1 — Cloner le dépôt

In [1]:
# Remplacez l'URL par celle de votre dépôt GitHub
GITHUB_URL = 'https://github.com/firminApp/cnn-catsdogs-BaniganteKpapou.git'

!git clone {GITHUB_URL} /content/repo
%cd /content/repo


Cloning into '/content/repo'...
remote: Enumerating objects: 31, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 31 (delta 10), reused 26 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (31/31), 1.88 MiB | 32.64 MiB/s, done.
Resolving deltas: 100% (10/10), done.
/content/repo


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Étape 2 — Installer les dépendances

In [3]:
!pip install -q -r requirements.txt


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 29.9 MB/s eta 0:00:00


## Étape 4 — Vérifier le GPU

In [4]:
import torch
print('CUDA disponible :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU :', torch.cuda.get_device_name(0))


CUDA disponible : True
GPU : Tesla T4


## Étape 3 — Monter Google Drive et extraire les données

In [5]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil

DRIVE_BASE = '/content/drive/MyDrive/notebooks/Deep_learning/deep_learning_cnn_cat_dog_classification/data'
ZIP_SRC    = DRIVE_BASE + '/cats_vs_dogs.zip'
SRC_FOLDER = DRIVE_BASE + '/cats_vs_dogs'
DATA_DST   = '/content/repo/data'
FINAL_DIR  = DATA_DST + '/cats_vs_dogs'

os.makedirs(DATA_DST, exist_ok=True)

# --- Étape 1 : créer le ZIP sur Drive si absent (une seule fois) ---
if not os.path.isfile(ZIP_SRC):
    if not os.path.isdir(SRC_FOLDER):
        print("Contenu de DRIVE_BASE :")
        for e in sorted(os.listdir(DRIVE_BASE)):
            print(" ", e)
        raise FileNotFoundError(
            f"Ni le ZIP ni le dossier source trouvé dans :\n  {DRIVE_BASE}\n"
            "Adaptez DRIVE_BASE ci-dessus.")
    print("ZIP absent — création en cours (une seule fois)…")
    !cd "{DRIVE_BASE}" && zip -r "{ZIP_SRC}" cats_vs_dogs
    print("ZIP créé :", ZIP_SRC)
else:
    print(f"ZIP trouvé ({os.path.getsize(ZIP_SRC)/1e6:.1f} MB) — extraction directe.")

# --- Étape 2 : extraire le ZIP en local (rapide) ---
!unzip -o "{ZIP_SRC}" -d "{DATA_DST}" | tail -3

# --- Étape 3 : corriger la structure si train/ et test/ sont à la racine ---
if not os.path.isdir(FINAL_DIR):
    os.makedirs(FINAL_DIR, exist_ok=True)
    for sub in ['train', 'test', 'valid']:
        src = os.path.join(DATA_DST, sub)
        if os.path.isdir(src):
            shutil.move(src, FINAL_DIR)
            print(f"Déplacé : {sub}/ → cats_vs_dogs/{sub}/")

print("Structure finale :", os.listdir(FINAL_DIR))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ZIP trouvé (581.8 MB) — extraction directe.
  inflating: /content/repo/data/__MACOSX/cats_vs_dogs/train/dog/._dog.1992.jpg  
  inflating: /content/repo/data/cats_vs_dogs/train/dog/dog.12412.jpg  
  inflating: /content/repo/data/__MACOSX/cats_vs_dogs/train/dog/._dog.12412.jpg  
Structure finale : ['test', 'train', '.DS_Store']


## Étape 5 — Lancer l'entraînement complet

In [6]:
# Expérience A — CNN scratch / Adam
!python train_cats_dogs.py \
  --data-dir data/cats_vs_dogs \
  --epochs 15 --optimizer adam --lr 1e-3 \
  --scheduler-step 5 --scheduler-gamma 0.5 \
  --checkpoint-dir checkpoints


Device: cuda
Classes: ['cat', 'dog']
Train samples: 20250
Valid samples: 2250
Test samples: 2500
Experiment A: CNN from scratch
Epoch 1/15 | train_loss=1.1106 val_loss=0.6441 acc=0.6231 prec=0.6509 recall=0.6231 lr=0.00100
Saved checkpoint: checkpoints/cnn_scratch_adam.pth
Epoch 2/15 | train_loss=0.6507 val_loss=0.6242 acc=0.6097 prec=0.7314 recall=0.6097 lr=0.00100
Saved checkpoint: checkpoints/cnn_scratch_adam.pth
Epoch 3/15 | train_loss=0.6396 val_loss=0.5983 acc=0.6583 prec=0.7322 recall=0.6583 lr=0.00100
Saved checkpoint: checkpoints/cnn_scratch_adam.pth
Epoch 4/15 | train_loss=0.6294 val_loss=0.5806 acc=0.6447 prec=0.7382 recall=0.6447 lr=0.00100
Saved checkpoint: checkpoints/cnn_scratch_adam.pth
Epoch 5/15 | train_loss=0.6234 val_loss=0.5689 acc=0.6889 prec=0.7136 recall=0.6889 lr=0.00050
Saved checkpoint: checkpoints/cnn_scratch_adam.pth
Epoch 6/15 | train_loss=0.6006 val_loss=0.5374 acc=0.7248 prec=0.7345 recall=0.7248 lr=0.00050
Saved checkpoint: checkpoints/cnn_scratch_adam.

In [7]:
# Expérience B — ResNet18 Transfer / Adam
!python train_cats_dogs.py \
  --data-dir data/cats_vs_dogs \
  --epochs 15 --optimizer adam --lr 1e-4 \
  --feature-extract \
  --scheduler-step 5 --scheduler-gamma 0.5 \
  --checkpoint-dir checkpoints


Device: cuda
Classes: ['cat', 'dog']
Train samples: 20250
Valid samples: 2250
Test samples: 2500
Experiment A: CNN from scratch
Epoch 1/15 | train_loss=0.7016 val_loss=0.6352 acc=0.6246 prec=0.6855 recall=0.6246 lr=0.00010
Saved checkpoint: checkpoints/cnn_scratch_adam.pth
Epoch 2/15 | train_loss=0.6196 val_loss=0.6032 acc=0.6676 prec=0.7278 recall=0.6676 lr=0.00010
Saved checkpoint: checkpoints/cnn_scratch_adam.pth
Epoch 3/15 | train_loss=0.5933 val_loss=0.5833 acc=0.6828 prec=0.7390 recall=0.6828 lr=0.00010
Saved checkpoint: checkpoints/cnn_scratch_adam.pth
Epoch 4/15 | train_loss=0.5737 val_loss=0.5634 acc=0.6948 prec=0.7556 recall=0.6948 lr=0.00010
Saved checkpoint: checkpoints/cnn_scratch_adam.pth
Epoch 5/15 | train_loss=0.5556 val_loss=0.5145 acc=0.7360 prec=0.7584 recall=0.7360 lr=0.00005
Saved checkpoint: checkpoints/cnn_scratch_adam.pth
Epoch 6/15 | train_loss=0.5368 val_loss=0.5205 acc=0.7289 prec=0.7592 recall=0.7289 lr=0.00005
Epoch 7/15 | train_loss=0.5214 val_loss=0.5105 

## (Optionnel) Test rapide — debug 1 époque

In [8]:
!python train_cats_dogs.py \
  --data-dir data/cats_vs_dogs \
  --epochs 1 \
  --max-train-batches 20 --max-valid-batches 5 --max-test-batches 5


Device: cuda
Classes: ['cat', 'dog']
Train samples: 20250
Valid samples: 2250
Test samples: 2500
Experiment A: CNN from scratch
Epoch 1/1 | train_loss=1.6213 val_loss=0.7349 acc=0.5000 prec=0.2469 recall=0.5000 lr=0.00010
Saved checkpoint: checkpoints/cnn_scratch_adam.pth
Evaluating best model for cnn_scratch...
Test | loss=0.4310 acc=1.0000 prec=1.0000 recall=1.0000
Confusion matrix:
[[160   0]
 [  0   0]]

Experiment B: Transfer learning with ResNet18
Epoch 1/1 | train_loss=0.2566 val_loss=0.0930 acc=0.9690 prec=0.9692 recall=0.9690 lr=0.00010
Saved checkpoint: checkpoints/resnet18_transfer_adam.pth
Evaluating best model for resnet18_transfer...
Test | loss=0.0572 acc=0.4906 prec=0.5000 recall=0.4906
Confusion matrix:
[[157   3]
 [  0   0]]
